In [18]:
"""
============================================================
SIMULATIE — THUISBATTERIJ PV-KLANTEN
Thesis: Meerwaarde thuisbatterij voor Vlaamse PV-huishoudens
============================================================
Berekent per PV-klant de jaarlijkse factuur en besparing
voor zeven scenario's, vergeleken met het referentiescenario
(vast contract, geen batterij).

SCENARIO'S:
  [A] Vast contract,      geen batterij  — referentie
  [B] Dynamisch contract, geen batterij
  [C] Vast contract,      batterij pieksturing    (perfect foresight)
  [D] Dynamisch contract, batterij prijsarbitrage (perfect foresight)
  [E] Dynamisch contract, batterij pieksturing    (perfect foresight)
  [F] Vast contract,      batterij SCM zelfconsumptie (realistisch)
  [G] Dynamisch contract, batterij SCM zelfconsumptie (realistisch)

C/D/E vereisen perfect foresight op de day-ahead prijzen van
de volgende dag en vormen een theoretische bovengrens.
F/G zijn realistisch: de batterij reageert enkel op het
actuele verbruik en de actuele productie zonder vooruitkijk.

TARIEVEN (Eneco tariefkaarten januari 2024, Vlaanderen, incl. 6% btw tenzij vermeld):

  VAST CONTRACT (portfoliomethode, consistent met simulatie.py hoofdstuk 3.3):
  Vast afname:       8,33 ct/kWh incl. btw (portfoliomethode baseline 2024)
  Vast injectie:     4,02 ct/kWh excl. btw (portfoliomethode baseline 2024)

  DYNAMISCH CONTRACT (Eneco Zon & Wind Dynamisch):
  Dyn. afname:       (0,102 × EPEX_MWh + 1,0) × 1,06 / 100  €/kWh
  Dyn. injectie:     (0,100 × EPEX_MWh − 1,188) / 100  €/kWh excl. btw
                     (kan negatief zijn bij EPEX < 11,88 EUR/MWh)

  NETCOMPONENT (identiek voor beide contracttypes, gem. 10 Vlaamse zones):
  Volumetrisch:      4,05 ct/kWh
  Capaciteitstarief: 44,00 €/kW/jaar (op gem. maandelijkse piekafname, min. 2,5 kW)
  Databeheer SMR1:   13,39 €/jaar (vast contract)
  Databeheer SMR3:   14,53 €/jaar (dynamisch contract)
  (Transportkosten niet van toepassing: enkel bij Waalse netbeheerders)

  HEFFINGEN (identiek voor beide contracttypes):
  Bijzondere accijns + energiebijdrage + bijdrage groene stroom: 6,7971 ct/kWh

BATTERIJPARAMETERS (Alpha-ESS Smile G3):
  Capaciteit:        9,3 kWh
  Max vermogen:      5,0 kW (laden en ontladen)
  Roundtrip eff.:    92%

INPUTBESTANDEN:
  fluvius_300_met_ZP_uur.csv
  fluvius_300_met_EV_met_ZP_uur.csv
  fluvius_300_met_WP_met_ZP_uur.csv
  fluvius_300_met_WP_met_EV_met_ZP_uur.csv
  Belgium_2024_MWh.csv

OUTPUTBESTAND:
  besparing_pv_batterij.csv
============================================================
"""

import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATIE
# ============================================================

FLUVIUS_BESTANDEN = [
    "fluvius_300_met_ZP_uur.csv",
    "fluvius_300_met_EV_met_ZP_uur.csv",
    "fluvius_300_met_WP_met_ZP_uur.csv",
    "fluvius_300_met_WP_met_EV_met_ZP_uur.csv",
]
EMBER_BESTAND  = "Belgium_2024_MWh.csv"
OUTPUT_BESTAND = "besparing_pv_batterij.csv"

# ── Vast contract — consistent met simulatie.py (portfoliomethode) ────────────
# Dezelfde tarieven als in simulatie.py, hier incl. 6% btw
# op afname zodat de vergelijking met dynamisch op klantperspectief correct is.
# Gebaseerd op de Vlaamse baseline 2024, berekend op alle 2.400 Fluvius-meters.
VAST_AFNAME_CT   = 8.33    # ct/kWh, incl. 6% btw (portfoliomethode baseline 2024)
VAST_INJECTIE_CT = 4.02    # ct/kWh, excl. btw (btw-vrijgesteld)

# ── Dynamisch contract (Eneco Zon & Wind Dynamisch, tariefkaart jan 2024) ────
BTW           = 1.06
DYN_ALPHA_AFN = 0.102
DYN_BETA_AFN  = 1.0         # ct/kWh excl. btw
DYN_ALPHA_INJ = 0.100
DYN_BETA_INJ  = 1.188       # ct/kWh, excl. btw
                              # injectievergoeding kan negatief zijn
                              # bij EPEX < 11,88 EUR/MWh

# ── Vaste vergoeding niet meegenomen ─────────────────────────────────────────
# De vaste vergoeding (65 €/jaar vast, 100 €/jaar dynamisch) is identiek
# voor alle scenario's binnen hetzelfde contracttype en heeft geen invloed
# op de besparing door een batterij. Ze wordt daarom weggelaten.

# Netcomponent (Fluvius, conform beide tariefkaarten jan 2024)
# Rekenkundig gemiddelde over de 10 Vlaamse Fluvius-zones
# Transportkosten worden NIET meegenomen: die staan enkel bij Waalse
# netbeheerders op de tariefkaart, niet bij de Fluvius-zones.
NET_VOL_CT   = 4.05         # ct/kWh, volumetrisch distributietarief
CAP_TARIEF   = 44.00        # €/kW/jaar, capaciteitstarief
CAP_MIN_KW   = 2.5          # kW minimum (contractueel ondergrens)

# Databeheerkosten (identiek op beide tariefkaarten)
DATABEHEER_SMR1 = 13.39     # €/jaar, SMR1-meetregime (vast contract)
DATABEHEER_SMR3 = 14.53     # €/jaar, SMR3-meetregime (dynamisch contract)

# ── Heffingen (overheid, identiek voor beide contracttypes) ──────────────────
HEFFINGEN_CT = 5.0329 + 0.2042 + 1.5600  # = 6.7971 ct/kWh (excl. transport)
                                            # bijzondere accijns (5,0329)
                                            # + energiebijdrage (0,2042)
                                            # + bijdrage groene stroom (1,5600)

# Batterijparameters (Alpha-ESS Smile G3)
BAT_CAP_KWH = 9.3          # kWh nominale capaciteit
BAT_MAX_KW  = 5.0          # kW maximaal laad- en ontlaadvermogen
BAT_EFF     = 0.92         # roundtrip efficiëntie (laden × ontladen)

# NCW-parameters financiële evaluatie
NCW_I0 = 3606.41   # aanschafprijs batterij incl. btw (Alpha-ESS Smile G3)
NCW_R  = 0.035     # discontovoet (risicovrije rente, Belgische lineaire obligaties 10j)
NCW_T  = 10        # simulatiehorizon in jaren (garantieperiode batterij)

# Leesbare profielnamen
PROFIEL_LABELS = {
    "met_ZP":               "ZP",
    "met_EV_met_ZP":        "ZP + EV",
    "met_WP_met_ZP":        "ZP + WP",
    "met_WP_met_EV_met_ZP": "ZP + WP + EV",
}
PROFIEL_VOLGORDE = ["ZP", "ZP + EV", "ZP + WP", "ZP + WP + EV"]


# ============================================================
# STAP 1 — DATA LADEN EN KOPPELEN
# ============================================================

def laad_data() -> pd.DataFrame:
    """
    Laadt de vier ZP-profielbestanden en koppelt de uurlijkse
    EPEX-groothandelsprijzen. De Fluvius-data wordt ingeladen als
    UTC en omgezet naar Belgische lokale tijd (CET/CEST) voor de
    koppeling met de Ember-prijsdata op maand/dag/uur in lokale tijd.
    Ontbrekende prijzen na de koppeling worden opgevuld met de mediane jaarprijs.
    """
    print(f"Fluvius: {len(FLUVIUS_BESTANDEN)} profielbestanden")
    print(f"Ember:   {EMBER_BESTAND}")

    # Laad Fluvius uurdata
    dfs = []
    for bestand in FLUVIUS_BESTANDEN:
        df = pd.read_csv(bestand)
        df["profiel"] = bestand.replace("fluvius_300_", "").replace("_uur.csv", "")
        dfs.append(df)
    fluvius = pd.concat(dfs, ignore_index=True)

    fluvius = fluvius.rename(columns={
        "EAN_ID":              "klant_id",
        "Datum_Startuur":      "timestamp",
        "Volume_Afname_KWh":   "afname_kwh",
        "Volume_Injectie_KWh": "injectie_kwh",
    })
    fluvius["timestamp"]    = pd.to_datetime(fluvius["timestamp"], utc=True)
    fluvius["afname_kwh"]   = pd.to_numeric(
        fluvius["afname_kwh"],   errors="coerce").fillna(0).clip(lower=0)
    fluvius["injectie_kwh"] = pd.to_numeric(
        fluvius["injectie_kwh"], errors="coerce").fillna(0).clip(lower=0)

    # Laad Ember prijsdata — staat al in lokale tijd (Datetime Local kolom)
    # Koppeling gebeurt op maand/dag/uur in lokale tijd, identiek aan simulatie.py
    ember = pd.read_csv(EMBER_BESTAND)
    ember = ember.rename(columns={
        "Datetime (Local)": "timestamp",
        "Price (EUR/MWh)":  "prijs_eur_mwh",
    })
    ember["timestamp"]    = pd.to_datetime(ember["timestamp"])
    ember["maand"] = ember["timestamp"].dt.month
    ember["dag"]   = ember["timestamp"].dt.day
    ember["uur"]   = ember["timestamp"].dt.hour

    # Wintertijdovergang: dubbel uur → gemiddelde
    ember = ember.groupby(["maand", "dag", "uur"]).agg(
        prijs_eur_mwh=("prijs_eur_mwh", "mean"),
    ).reset_index()

    # Koppeling via maand/dag/uur in lokale tijd
    fluvius["timestamp_lokaal"] = pd.to_datetime(
        fluvius["timestamp"], utc=True
    ).dt.tz_convert("Europe/Brussels").dt.tz_localize(None)
    fluvius["maand"] = fluvius["timestamp_lokaal"].dt.month
    fluvius["dag"]   = fluvius["timestamp_lokaal"].dt.day
    fluvius["uur"]   = fluvius["timestamp_lokaal"].dt.hour

    # Verwijder NaT (zomertijdovergang 27 mrt 02:00 bestaat niet in lokale tijd)
    fluvius = fluvius.dropna(subset=["timestamp_lokaal"]).copy()

    data = fluvius.merge(
        ember[["maand", "dag", "uur", "prijs_eur_mwh"]],
        on=["maand", "dag", "uur"], how="left"
    )

    # Vul ontbrekende prijzen op met mediane jaarprijs
    n_missing = data["prijs_eur_mwh"].isna().sum()
    if n_missing > 0:
        print(f"  {n_missing} ontbrekende prijzen → mediaan gebruikt")
    data["prijs_eur_mwh"] = data["prijs_eur_mwh"].fillna(
        data["prijs_eur_mwh"].median()
    )

    # Profiellabel toevoegen
    data["profiel_label"] = data["profiel"].map(PROFIEL_LABELS).fillna(data["profiel"])

    print(f"\nGekoppeld: {data['klant_id'].nunique()} klanten | {len(data):,} rijen")
    return data


# ============================================================
# STAP 2 — CAPACITEITSTARIEF BEREKENEN
# ============================================================

def bereken_facturatiepiek(data: pd.DataFrame) -> dict:
    """
    Berekent het capaciteitstarief per klant op basis van de gemiddelde
    maandelijkse piekafname over de twaalf kalendermaanden.

    Het capaciteitstarief bedraagt 44,00 €/kW/jaar, berekend op de
    hoogste uurafname per kalendermaand, gemiddeld over de twaalf maanden.
    Het contractuele minimum is 2,5 kW.

    Returns: dict {klant_id: gemiddelde maandpiek in kW}
    """
    df = data[["klant_id", "timestamp", "afname_kwh"]].copy()
    df["maand"] = df["timestamp"].dt.strftime("%Y-%m")

    # Hoogste uurafname per klant per maand
    maandpiek = df.groupby(["klant_id", "maand"])["afname_kwh"].max()

    # Gemiddelde over de twaalf maanden, met contractueel minimum van 2,5 kW
    return maandpiek.groupby("klant_id").mean().clip(lower=CAP_MIN_KW).to_dict()


# ============================================================
# STAP 3 — FACTUURBEREKENINGEN
# ============================================================

def bereken_factuur_vast(afname: np.ndarray, injectie: np.ndarray,
                          piek: float) -> dict:
    """
    Berekent de volledige jaarfactuur op een vast contract.

    Gebruikt dezelfde portfoliotarieven als simulatie.py (hoofdstuk 3.3),
    aangevuld met 6% btw op afname en de werkelijke netcomponent en heffingen
    conform de Eneco tariefkaart januari 2024.

    Afname:  8,33 ct/kWh incl. btw (portfoliomethode baseline 2024)
    Injectie: 4,02 ct/kWh excl. btw (portfoliomethode baseline 2024)

    Returns: dict met totaal en afzonderlijke componenten
    """
    tot_afn = afname.sum()
    tot_inj = injectie.sum()

    energie   = (tot_afn * VAST_AFNAME_CT - tot_inj * VAST_INJECTIE_CT) / 100
    net_vol   = tot_afn * NET_VOL_CT / 100
    cap       = piek * CAP_TARIEF
    heffingen = tot_afn * HEFFINGEN_CT / 100

    return {
        "totaal":      energie + net_vol + cap + heffingen + DATABEHEER_SMR1,
        "energie":     energie,
        "net_vol":     net_vol,
        "cap":         cap,
        "heffingen":   heffingen,
        "databeheer":  DATABEHEER_SMR1,
        "afname_kwh":  tot_afn,
        "injectie_kwh":tot_inj,
    }


def bereken_factuur_dynamisch(afname: np.ndarray, injectie: np.ndarray,
                               prijs_mwh: np.ndarray, piek: float) -> dict:
    """
    Berekent de volledige jaarfactuur op het Eneco Zon & Wind Dynamisch contract
    (tariefkaart januari 2024).

    Afnametarief per uur (incl. 6% btw):
      T_afname,u = (0,102 × EPEX_u + 1,0) × 1,06 / 100  (€/kWh)

    Injectievergoeding per uur (excl. btw, kan negatief zijn conform tariefkaart):
      T_injectie,u = (0,100 × EPEX_u − 1,188) / 100  (€/kWh)
      Bij EPEX < 11,88 EUR/MWh betaalt de klant bij voor injectie.
      Er is geen contractueel minimum — de tariefkaart vermeldt geen begrenzing.

    Returns: dict met totaal en afzonderlijke componenten
    """
    tarief_afn = (DYN_ALPHA_AFN * prijs_mwh + DYN_BETA_AFN) * BTW  # ct/kWh
    tarief_inj = DYN_ALPHA_INJ * prijs_mwh - DYN_BETA_INJ           # ct/kWh, kan negatief conform tariefkaart

    energie   = ((afname * tarief_afn).sum() - (injectie * tarief_inj).sum()) / 100
    tot_afn   = afname.sum()
    tot_inj   = injectie.sum()
    net_vol   = tot_afn * NET_VOL_CT / 100
    cap       = piek * CAP_TARIEF
    heffingen = tot_afn * HEFFINGEN_CT / 100

    return {
        "totaal":      energie + net_vol + cap + heffingen + DATABEHEER_SMR3,
        "energie":     energie,
        "net_vol":     net_vol,
        "cap":         cap,
        "heffingen":   heffingen,
        "databeheer":  DATABEHEER_SMR3,
        "afname_kwh":  tot_afn,
        "injectie_kwh":tot_inj,
    }


# ============================================================
# STAP 4 — BATTERIJSTRATEGIEËN
# ============================================================

def simuleer_batterij_scm(klant_data: pd.DataFrame) -> tuple:
    """
    Self-Consumption Maximization (SCM) — realistische strategie.

    De batterij laadt op PV-overschot (injectie) en ontlaadt bij
    netafname, chronologisch over het volledige jaar zonder vooruitkijk.
    Lading kan van dag naar dag worden meegenomen. De round-trip
    efficiëntie van 92% wordt toegepast bij het ontladen: de batterij
    levert minder dan wat er werd opgeslagen.

    Dit is de meest realistische strategie: geen kennis van toekomstige
    prijzen of verbruiken vereist, enkel het actuele moment.

    Returns: (afname_na, injectie_na) — aangepaste uurprofielen
    """
    df = klant_data.sort_values("timestamp").copy()
    afname_na   = df["afname_kwh"].values.copy().astype(float)
    injectie_na = df["injectie_kwh"].values.copy().astype(float)
    opslag = 0.0

    for t in range(len(df)):
        # Laden op PV-overschot — geen efficiëntieverliezen bij laden
        inj_t = injectie_na[t]
        if inj_t > 0:
            laden = min(inj_t, BAT_MAX_KW, BAT_CAP_KWH - opslag)
            laden = max(0.0, laden)
            injectie_na[t] = max(0.0, inj_t - laden)
            opslag += laden

        # Ontladen bij netafname — efficiëntieverliezen bij ontladen
        if afname_na[t] > 0 and opslag > 0:
            ontladen_uit_bat  = min(afname_na[t] / BAT_EFF, BAT_MAX_KW, opslag)
            ontladen_uit_bat  = max(0.0, ontladen_uit_bat)
            geleverd_aan_huis = ontladen_uit_bat * BAT_EFF
            afname_na[t] = max(0.0, afname_na[t] - geleverd_aan_huis)
            opslag -= ontladen_uit_bat

    return afname_na, injectie_na


def simuleer_batterij_foresight(klant_data: pd.DataFrame,
                                 strategie: str = "prijsarbitrage") -> tuple:
    """
    Perfect foresight strategie — theoretische bovengrens.

    Laden: het PV-overschot van dag D wordt volledig opgeslagen.
    Ontladen: de opgeslagen energie van dag D wordt de volgende dag
    (dag D+1) ontladen op de uren met de hoogste prijs (prijsarbitrage)
    of de hoogste afname (pieksturing).

    Deze strategie vereist perfecte kennis van de prijzen en verbruiken
    van de volgende dag en is daardoor niet realistisch inzetbaar.
    Ze geeft een theoretische bovengrens voor de batterijmeerwaarde.

    Parameters:
      strategie: "prijsarbitrage" → ontlaad op duurste uren (scenario D)
                 "pieksturing"    → ontlaad op hoogste afname (scenario C/E)

    Returns: (afname_na, injectie_na) — aangepaste uurprofielen
    """
    df = klant_data.sort_values("timestamp").copy()
    df["datum"] = df["timestamp"].dt.strftime("%Y-%m-%d")

    afname_na   = df["afname_kwh"].values.copy().astype(float)
    injectie_na = df["injectie_kwh"].values.copy().astype(float)

    datums     = sorted(df["datum"].unique())
    opgeslagen = 0.0   # dag 1: lege batterij

    for dag in datums:
        mask = df["datum"].values == dag
        idx  = np.where(mask)[0]

        prijs = df["prijs_eur_mwh"].values[idx]
        afn   = df["afname_kwh"].values[idx]
        inj   = df["injectie_kwh"].values[idx]

        # Ontladen: gebruik energie opgeslagen van gisteren
        if opgeslagen > 0:
            # Sorteer uren op basis van gekozen strategie
            if strategie == "prijsarbitrage":
                volgorde = np.argsort(prijs)[::-1]   # duurste uren eerst
            else:
                volgorde = np.argsort(afn)[::-1]     # hoogste afname eerst

            soc = opgeslagen
            for t in volgorde:
                if soc <= 0:
                    break
                # Hoeveel kan de batterij maximaal leveren aan het huis?
                # Begrensd door vermogen, lading en de actuele afname
                ontladen_uit_bat  = min(BAT_MAX_KW, soc, max(0.0, afn[t]) / BAT_EFF)
                geleverd_aan_huis = ontladen_uit_bat * BAT_EFF
                afname_na[idx[t]] = max(0.0, afn[t] - geleverd_aan_huis)
                soc -= ontladen_uit_bat

        # Laden: sla PV-overschot van vandaag op voor morgen
        # Per uur begrensd door maximaal laadvermogen (5 kW) en resterende capaciteit
        pv_overschot = 0.0
        opslag_tmp   = 0.0
        for t in range(len(idx)):
            if inj[t] > 0:
                laden = min(inj[t], BAT_MAX_KW, BAT_CAP_KWH - opslag_tmp)
                laden = max(0.0, laden)
                opslag_tmp   += laden
                pv_overschot += laden

        opgeslagen = opslag_tmp

        if pv_overschot > 0 and opgeslagen > 0:
            fractie = opgeslagen / pv_overschot if pv_overschot > 0 else 0.0
            for t in range(len(idx)):
                if inj[t] > 0:
                    injectie_na[idx[t]] = max(0.0, inj[t] * (1 - fractie))

    return afname_na, injectie_na


# ============================================================
# HULPFUNCTIE: PIEKAFNAME NA BATTERIJ
# ============================================================

def bereken_piek_na_batterij(afname_na: np.ndarray,
                              maand: np.ndarray) -> float:
    """
    Herberekent de gemiddelde maandelijkse piekafname na batterijinzet.
    Wordt gebruikt voor het capaciteitstarief in scenario's met batterij.
    """
    df_piek = pd.DataFrame({"afname": afname_na, "maand": maand})
    piek    = df_piek.groupby("maand")["afname"].max().mean(skipna=True)
    return max(piek, CAP_MIN_KW)


# ============================================================
# HOOFDPROGRAMMA
# ============================================================

def main():
    print("=" * 70)
    print("SIMULATIE — THUISBATTERIJ PV-KLANTEN")
    print("Scenario's: A (vast) | B (dyn) | C/D/E (foresight) | F/G (SCM)")
    print("=" * 70)
    print(f"\nBatterij: {BAT_CAP_KWH} kWh / {BAT_MAX_KW} kW / {BAT_EFF*100:.0f}% eff.")
    print(f"\nVast contract (portfoliomethode, consistent met simulatie.py):")
    print(f"  Afname:          {VAST_AFNAME_CT} ct/kWh incl. btw")
    print(f"  Injectie:        {VAST_INJECTIE_CT} ct/kWh excl. btw")
    print(f"\nDynamisch contract (Eneco Zon & Wind Dynamisch, jan 2024):")
    print(f"  Afname:          (0,102 × EPEX + 1,0) × 1,06 / 100 €/kWh")
    print(f"  Injectie:        (0,100 × EPEX − 1,188) / 100 €/kWh (kan negatief)")
    print(f"\nNetcomponent (beide contracten):")
    print(f"  Volumetrisch:    {NET_VOL_CT} ct/kWh")
    print(f"  Capaciteit:      {CAP_TARIEF} €/kW/jaar (min. {CAP_MIN_KW} kW)")
    print(f"  Databeheer SMR1: {DATABEHEER_SMR1} €/jaar")
    print(f"  Databeheer SMR3: {DATABEHEER_SMR3} €/jaar")
    print(f"  Heffingen:       {HEFFINGEN_CT:.4f} ct/kWh")

    # Stap 1: Data laden
    print("\n1. Data inladen...")
    data = laad_data()
    profiel_map = data.groupby("klant_id")["profiel"].first().to_dict()
    klanten     = data["klant_id"].unique()

    # Stap 2: Capaciteitspieken zonder batterij
    print("\n2. Facturatiepieken berekenen...")
    pieken = bereken_facturatiepiek(data)
    print(f"  Gemiddelde facturatiepiek: {sum(pieken.values())/len(pieken):.2f} kW")

    # Stap 3: Simulatie per klant
    print(f"\n3. Simulatie per klant ({len(klanten)} klanten)...")
    resultaten = []

    for i, klant_id in enumerate(klanten):
        if i % 200 == 0:
            print(f"   {i+1}/{len(klanten)}...")

        kd    = data[data["klant_id"] == klant_id].copy()
        afn   = kd["afname_kwh"].values
        inj   = kd["injectie_kwh"].values
        prijs = kd["prijs_eur_mwh"].values
        maand = kd["timestamp"].dt.strftime("%Y-%m").values
        piek  = pieken.get(klant_id, CAP_MIN_KW)

        # [A] Vast, geen batterij — referentie
        fA = bereken_factuur_vast(afn, inj, piek)

        # [B] Dynamisch, geen batterij
        fB = bereken_factuur_dynamisch(afn, inj, prijs, piek)

        # [C] Vast + batterij, pieksturing (perfect foresight)
        afn_C, inj_C = simuleer_batterij_foresight(kd, strategie="pieksturing")
        piek_C = bereken_piek_na_batterij(afn_C, maand)
        fC = bereken_factuur_vast(afn_C, inj_C, piek_C)

        # [D] Dynamisch + batterij, prijsarbitrage (perfect foresight)
        afn_D, inj_D = simuleer_batterij_foresight(kd, strategie="prijsarbitrage")
        piek_D = bereken_piek_na_batterij(afn_D, maand)
        fD = bereken_factuur_dynamisch(afn_D, inj_D, prijs, piek_D)

        # [E] Dynamisch + batterij, pieksturing (perfect foresight)
        afn_E, inj_E = simuleer_batterij_foresight(kd, strategie="pieksturing")
        piek_E = bereken_piek_na_batterij(afn_E, maand)
        fE = bereken_factuur_dynamisch(afn_E, inj_E, prijs, piek_E)

        # [F] Vast + batterij, SCM zelfconsumptie (realistisch)
        afn_F, inj_F = simuleer_batterij_scm(kd)
        piek_F = bereken_piek_na_batterij(afn_F, maand)
        fF = bereken_factuur_vast(afn_F, inj_F, piek_F)

        # [G] Dynamisch + batterij, SCM zelfconsumptie (realistisch)
        # Gebruikt dezelfde aangepaste profielen als F
        fG = bereken_factuur_dynamisch(afn_F, inj_F, prijs, piek_F)

        resultaten.append({
            "klant_id":      klant_id,
            "profiel":       profiel_map.get(klant_id, "onbekend"),
            "profiel_label": PROFIEL_LABELS.get(profiel_map.get(klant_id, ""), "onbekend"),
            # Jaarfacturen
            "factuur_A":     round(fA["totaal"], 2),
            "factuur_B":     round(fB["totaal"], 2),
            "factuur_C":     round(fC["totaal"], 2),
            "factuur_D":     round(fD["totaal"], 2),
            "factuur_E":     round(fE["totaal"], 2),
            "factuur_F":     round(fF["totaal"], 2),
            "factuur_G":     round(fG["totaal"], 2),
            # Besparingen t.o.v. referentie A
            "besp_B":        round(fA["totaal"] - fB["totaal"], 2),
            "besp_C":        round(fA["totaal"] - fC["totaal"], 2),
            "besp_D":        round(fA["totaal"] - fD["totaal"], 2),
            "besp_E":        round(fA["totaal"] - fE["totaal"], 2),
            "besp_F":        round(fA["totaal"] - fF["totaal"], 2),
            "besp_G":        round(fA["totaal"] - fG["totaal"], 2),
            # Piekafnames voor capaciteitstarief
            "piek_A":        round(piek,   2),
            "piek_C":        round(piek_C, 2),
            "piek_D":        round(piek_D, 2),
            "piek_E":        round(piek_E, 2),
            "piek_F":        round(piek_F, 2),
            # Verbruiksvolumes (zonder batterij)
            "afname_kwh":    round(afn.sum(), 2),
            "injectie_kwh":  round(inj.sum(), 2),
        })

    res = pd.DataFrame(resultaten)

    # NCW berekening
    # NCW = -I0 + B × [(1+r)^t - 1] / [r × (1+r)^t]
    I0 = NCW_I0
    R  = NCW_R
    T  = NCW_T
    AF = ((1 + R)**T - 1) / (R * (1 + R)**T)  # annuïteitsfactor

    for scenario in ["C", "D", "E", "F", "G"]:
        res[f"ncw_{scenario}"] = (-I0 + res[f"besp_{scenario}"] * AF).round(2)

    print(f"\nNCW-parameters: I0=€{I0:,.2f} | r={R*100:.1f}% | t={T} jaar")
    print(f"Annuïteitsfactor: {AF:.4f} | Break-even besparing: €{I0/AF:.0f}/jaar")

    # Overzicht resultaten
    print("\n" + "=" * 70)
    print("OVERZICHT RESULTATEN")
    print("=" * 70)

    for profiel in PROFIEL_VOLGORDE:
        groep = res[res["profiel_label"] == profiel]
        if len(groep) == 0:
            continue
        print(f"\n── {profiel} ({len(groep)} klanten) ──")
        print(f"  [A] Vast, geen batterij:                €{groep['factuur_A'].mean():>8.2f}/jaar")
        print(f"  [B] Dynamisch, geen batterij:           €{groep['factuur_B'].mean():>8.2f}/jaar")
        print(f"  [C] Vast + batterij (pieksturing):      €{groep['factuur_C'].mean():>8.2f}/jaar")
        print(f"  [D] Dyn  + batterij (prijsarbitrage):   €{groep['factuur_D'].mean():>8.2f}/jaar")
        print(f"  [E] Dyn  + batterij (pieksturing):      €{groep['factuur_E'].mean():>8.2f}/jaar")
        print(f"  [F] Vast + batterij (SCM):              €{groep['factuur_F'].mean():>8.2f}/jaar")
        print(f"  [G] Dyn  + batterij (SCM):              €{groep['factuur_G'].mean():>8.2f}/jaar")
        print(f"  ---")
        ref = groep['factuur_A'].mean()
        for col, naam in [("besp_B","B"),("besp_C","C"),("besp_D","D"),
                          ("besp_E","E"),("besp_F","F"),("besp_G","G")]:
            gem = groep[col].mean()
            pct = gem / ref * 100
            pct_pos = (groep[col] > 0).mean() * 100
            print(f"  Besparing {naam} vs A: €{gem:>7.2f}/jaar ({pct:.1f}%) | {pct_pos:.0f}% klanten positief")

        print(f"  --- NCW (r={R*100:.1f}%, t={T}j, I0=€{I0:,.0f}) ---")
        for col, naam in [("ncw_C","C"),("ncw_D","D"),("ncw_E","E"),
                          ("ncw_F","F"),("ncw_G","G")]:
            gem_ncw = groep[col].mean()
            pct_pos = (groep[col] > 0).mean() * 100
            print(f"  NCW {naam}: €{gem_ncw:>7.0f} gem. | {pct_pos:.0f}% klanten positief")

    res.to_csv(OUTPUT_BESTAND, index=False)
    print(f"\n✓ Opgeslagen: {OUTPUT_BESTAND}")
    return res


if __name__ == "__main__":
    resultaten = main()

SIMULATIE — THUISBATTERIJ PV-KLANTEN
Scenario's: A (vast) | B (dyn) | C/D/E (foresight) | F/G (SCM)

Batterij: 9.3 kWh / 5.0 kW / 92% eff.

Vast contract (portfoliomethode, consistent met simulatie.py):
  Afname:          8.33 ct/kWh incl. btw
  Injectie:        4.02 ct/kWh excl. btw

Dynamisch contract (Eneco Zon & Wind Dynamisch, jan 2024):
  Afname:          (0,102 × EPEX + 1,0) × 1,06 / 100 €/kWh
  Injectie:        (0,100 × EPEX − 1,188) / 100 €/kWh (kan negatief)

Netcomponent (beide contracten):
  Volumetrisch:    4.05 ct/kWh
  Capaciteit:      44.0 €/kW/jaar (min. 2.5 kW)
  Databeheer SMR1: 13.39 €/jaar
  Databeheer SMR3: 14.53 €/jaar
  Heffingen:       6.7971 ct/kWh

1. Data inladen...
Fluvius: 4 profielbestanden
Ember:   Belgium_2024_MWh.csv

Gekoppeld: 1200 klanten | 10,539,600 rijen

2. Facturatiepieken berekenen...
  Gemiddelde facturatiepiek: 4.45 kW

3. Simulatie per klant (1200 klanten)...
   1/1200...
   201/1200...
   401/1200...
   601/1200...
   801/1200...
   1001/1